# Day 5: 模型架构、MoE 和 MLA 数据流

这是配置推导和教学级路由实验，不是生产 MoE/MLA 实现。
配套阅读 `docs/DAY05_ARCHITECTURES.md`。

In [ ]:
import torch
from reasoning_from_scratch.qwen3 import QWEN_CONFIG_06_B

cfg = QWEN_CONFIG_06_B
q_width = cfg["n_heads"] * cfg["head_dim"]
kv_width = cfg["n_kv_groups"] * cfg["head_dim"]
print("Qwen3 0.6B")
print("embedding width:", cfg["emb_dim"])
print("Q projection width:", q_width)
print("compact K/V projection width:", kv_width)
print("GQA sharing ratio:", cfg["n_heads"] / cfg["n_kv_groups"])
print("FFN:", cfg["emb_dim"], "->", cfg["hidden_dim"], "->", cfg["emb_dim"])
assert q_width == 2048
assert kv_width == 1024
assert cfg["n_heads"] / cfg["n_kv_groups"] == 2

## Dense FFN 与教学级 MoE

Dense FFN 每个 token 调用同一个 FFN；MoE 先由 Router 选择 top-k experts，再只调用选中的 experts。

In [ ]:
torch.manual_seed(7)
tokens, hidden, experts, top_k = 32, 8, 4, 2
x = torch.randn(tokens, hidden)
router = torch.nn.Linear(hidden, experts, bias=False)
router_logits = router(x)
scores, expert_ids = router_logits.topk(top_k, dim=-1)
weights = scores.softmax(dim=-1)
counts = torch.bincount(expert_ids.reshape(-1), minlength=experts)
print("router logits:", tuple(router_logits.shape))
print("selected expert ids:", tuple(expert_ids.shape))
print("selected expert weights:", tuple(weights.shape))
print("tokens dispatched per expert:", counts.tolist())
print("total dispatch slots:", int(counts.sum()), "expected:", tokens * top_k)
assert int(counts.sum()) == tokens * top_k

## MLA 教学图

当前 Notebook 不实现 MLA kernel，只记录阅读时需要追踪的张量。
先看 [MLA 直观讲义](../docs/MLA_INTUITION.md)，再运行 [两条计算路径对齐](day05_mla_intuition.ipynb)。
MLA 为每层每个历史 token 保存短 latent，各头用不同投影使用它，位置 Key 另存；不是整段历史只有一条摘要。

In [ ]:
print(r'''
x
├─ query path ───────────────→ query-related representation
└─ kv path → latent cKV ─────→ decomposed/absorbed K/V representation
                         └──→ position encoding path (version-specific)
''')
print("Questions to answer from the target implementation:")
for question in (
    "What latent state is cached during decode?",
    "Where is RoPE applied or decomposed?",
    "Which matrices can be absorbed into the query path?",
    "What is the actual cache shape?",
):
    print("-", question)

## Day 5 自测

- 为什么 Qwen3 的 Q projection width 是 2048，但 compact K/V width 是 1024？
- 如果 32 个 token、4 个 experts、top-k=2，dispatch slot 为什么是 64？
- expert load 最大的那个 expert 为什么可能决定这轮 dispatch 的速度？
- MLA 为什么不能只解释为“把 KV groups 改少”？
- 你的配置表中哪些内容是源码实测，哪些内容来自论文阅读？